# Cubic Zirconia Price Modeling

**Recruiter-facing end-to-end analysis · Supervised regression · Python 3.12/3.13**

> Histogram gradient boosting reaches untouched-test R² 0.981, RMSE 563, and 90.6% coverage for the training-calibrated 90% diagnostic interval.

## Executive summary

**Objective:** Compare interpretable and nonlinear price models, diagnose residual risk, and quantify empirical prediction coverage.

**Data:** 26,967 cubic-zirconia records with physical measurements, quality grades, and price.

**Verified result:** Histogram gradient boosting reaches untouched-test R² 0.981, RMSE 563, and 90.6% coverage for the training-calibrated 90% diagnostic interval.

**Decision supported:** Estimate price ranges and identify where prediction errors become operationally material.

The figures, tables, metrics, and execution counts in this notebook are saved outputs from the bundled data.

## 1. Business understanding

**Primary user:** A merchandising, pricing, or appraisal analyst.

**Decision:** Estimate price ranges and identify where prediction errors become operationally material.

**Why it matters:** A technically accurate result is useful only when its error costs, uncertainty, and decision boundary are visible. This project stays within the evidence available in the source data.

## 2. Analytical objective and success criteria

Technical success requires portable execution, explicit data-quality evidence, a justified baseline, leakage-safe validation, task-appropriate metrics, diagnostics, and saved artifacts. Business success requires a specific recommendation supported by the observed result without invented financial impact.

## 3. Reproducible environment

In [1]:
from pathlib import Path
import hashlib, importlib.util, json, os, platform, tempfile, time
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "portfolio-matplotlib-cache"))
import matplotlib, numpy as np, pandas as pd, scipy, sklearn
SLUG = '05-gem-price-regression'
def locate_project():
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        candidate = base if base.name == SLUG else base / "projects" / SLUG
        if (candidate / "src" / "analysis.py").is_file(): return candidate
    raise FileNotFoundError(SLUG)
PROJECT_ROOT = locate_project(); DATA_DIR = PROJECT_ROOT / "data"; REPORTS_DIR = PROJECT_ROOT / "reports"
print(pd.Series({"Python": platform.python_version(), "pandas": pd.__version__, "NumPy": np.__version__, "SciPy": scipy.__version__, "scikit-learn": sklearn.__version__, "Matplotlib": matplotlib.__version__}, name="version").to_string())
print(f"\nProject: {PROJECT_ROOT.name}")

Python          3.12.10
pandas            2.3.3
NumPy             2.5.2
SciPy            1.18.0
scikit-learn      1.9.0
Matplotlib       3.11.1

Project: 05-gem-price-regression


## 4. Data provenance and scope

Bundled in the original repository; market period, price units, sampling context, and redistribution terms are not fully documented.

The next cells expose exact files, byte sizes, checksums, schemas, and sample records.

### 4.1 Source-file inventory

In [2]:
rows=[]
for path in sorted(DATA_DIR.iterdir()):
    if path.is_file() and path.name != "README.md": rows.append({"file": path.name, "size_mb": round(path.stat().st_size/1_000_000,3), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:16]})
inventory=pd.DataFrame(rows); print(inventory.to_string(index=False))

              file  size_mb           sha256
cubic_zirconia.csv    1.399 05066fa8c4a52399


### 4.2 Raw-record and schema preview

In [3]:
def preview(path):
    if path.suffix.lower()==".csv": return pd.read_csv(path, nrows=5)
    if path.suffix.lower()==".xlsx": return pd.read_excel(path, nrows=5)
    return None
for path in sorted(DATA_DIR.iterdir()):
    frame=preview(path)
    if frame is None: continue
    for column in frame.select_dtypes(include="object"): frame[column]=frame[column].astype(str).str.replace(r"\\s+"," ",regex=True).str.slice(0,100)
    print(f"\n{path.name}: {frame.shape[1]} columns"); print(frame.to_string(index=False,max_cols=12))


cubic_zirconia.csv: 11 columns
 Unnamed: 0  carat       cut color clarity  depth  table    x    y    z  price
          1   0.30     Ideal     E     SI1   62.1     58 4.27 4.29 2.66    499
          2   0.33   Premium     G      IF   60.8     58 4.42 4.46 2.70    984
          3   0.90 Very Good     E    VVS2   62.2     60 6.04 6.12 3.78   6289
          4   0.42     Ideal     F     VS1   61.6     56 4.82 4.80 2.96   1082
          5   0.31     Ideal     F    VVS1   60.4     59 4.35 4.43 2.65    779


## 5. Data-quality assessment

The pipeline checks missingness, duplicates, invalid fields, identifiers, cardinality, and problem-specific leakage or chronology risks. No row is silently removed.

## 6. Reusable implementation

Large functions are kept in source code so the notebook remains a readable analytical narrative.

In [4]:
source_path=PROJECT_ROOT/"src"/"analysis.py"; text=source_path.read_text(encoding="utf-8")
print(f"Reusable implementation: {len(text.splitlines())} lines")
print("Functions:", ", ".join(line.split("(")[0].replace("def ","").strip() for line in text.splitlines() if line.startswith("def ")))

Reusable implementation: 130 lines
Functions: _pipeline, run_analysis


## 7. Methodology and hypotheses

Leakage-safe imputation/encoding, median baseline, linear and regularized log-target models, random forest, histogram boosting, cross-validation, residual diagnostics, permutation importance, segment errors, and calibration-residual intervals.

The central hypothesis is that the audited features or group structure contain decision-relevant signal beyond the documented baseline. Exploratory findings are not presented as causal effects.

## 8. Execute the complete pipeline

This cell reruns cleaning, feature engineering, model/statistical analysis, validation, tables, figures, and model artifacts.

In [5]:
spec=importlib.util.spec_from_file_location("rebuilt_05_gem_price_regression", PROJECT_ROOT/"src"/"analysis.py")
analysis=importlib.util.module_from_spec(spec); spec.loader.exec_module(analysis)
started=time.perf_counter(); results=analysis.run_analysis(); runtime=time.perf_counter()-started
assert results["status"]=="passed"
print(f"Pipeline status: {results['status']}\nRuntime: {runtime:.2f} seconds")

Pipeline status: passed
Runtime: 5.23 seconds


## 9. Executed data-quality evidence

In [6]:
for path in sorted((REPORTS_DIR/"tables").glob("*data_quality.csv")):
    frame=pd.read_csv(path); print(f"\n{path.name} ({len(frame)} fields)"); print(frame.to_string(index=False,max_rows=30))


data_quality.csv (11 fields)
    column   dtype  missing_count  missing_percent  unique_values  constant
Unnamed: 0   int64              0            0.000          26967     False
     carat float64              0            0.000            257     False
       cut  object              0            0.000              5     False
     color  object              0            0.000              7     False
   clarity  object              0            0.000              8     False
     depth float64            697            2.585            170     False
     table float64              0            0.000            112     False
         x float64              0            0.000            531     False
         y float64              0            0.000            526     False
         z float64              0            0.000            356     False
     price   int64              0            0.000           8742     False


## 10. Baseline, candidates, and primary result

In [7]:
primary=REPORTS_DIR/"tables"/'model_comparison.csv'
frame=pd.read_csv(primary); print(f"Primary evidence: {primary.name}, shape={frame.shape}")
print(frame.head(15).round(4).to_string(index=False))
print("\nVerified result:\n" + 'Histogram gradient boosting reaches untouched-test R² 0.981, RMSE 563, and 90.6% coverage for the training-calibrated 90% diagnostic interval.')

Primary evidence: model_comparison.csv, shape=(5, 5)
                 model  cv_rmse_mean  cv_rmse_std  cv_mae_mean  cv_r_squared_mean
hist_gradient_boosting      573.3211      30.0947     299.7832             0.9795
         random_forest      594.7283      27.0487     298.8835             0.9779
      log_target_ridge      921.8108      81.0471     443.5555             0.9467
     linear_regression     1128.5471      31.8824     742.0371             0.9205
       median_baseline     4293.9194      49.7442    2815.6583            -0.1497

Verified result:
Histogram gradient boosting reaches untouched-test R² 0.981, RMSE 563, and 90.6% coverage for the training-calibrated 90% diagnostic interval.


## 11. Validation, diagnostics, and robustness

In [8]:
sections=[key for key in ["validation","model_selection","tuning","residual_diagnostics","outlier_sensitivity","participant_bootstrap_intervals","diagnostic_90_percent_interval","empirical_90_percent_interval"] if key in results]
for key in sections: print(f"\n{key.upper()}\n"+json.dumps(results[key],indent=2)[:6000])


VALIDATION
{
  "training_rows": 21573,
  "untouched_test_rows": 5394,
  "model_selection": "4-fold shuffled CV on training only",
  "calibration_rows_for_interval": 4315,
  "random_seed": 42
}

RESIDUAL_DIAGNOSTICS
{
  "mean_residual": -6.671122622690334,
  "residual_skewness": -0.7602333770924546,
  "breusch_pagan_proxy_spearman_abs_residual_vs_prediction": 0.6262986222080734
}

EMPIRICAL_90_PERCENT_INTERVAL
{
  "absolute_residual_radius": 750.3671880805578,
  "test_coverage": 0.9056358917315536,
  "method": "training calibration residual quantile; constant-width diagnostic interval"
}


## 12. Visual evidence

### Gem Price Model Evidence

![gem_price_model_evidence](../reports/figures/gem_price_model_evidence.png)

### Test Residuals

![test_residuals](../reports/figures/test_residuals.png)

## 13. Business interpretation

Histogram gradient boosting reaches untouched-test R² 0.981, RMSE 563, and 90.6% coverage for the training-calibrated 90% diagnostic interval.

The correct action is to use this result as evidence for **Estimate price ranges and identify where prediction errors become operationally material.**, while retaining the documented baseline and monitoring the error or sensitivity segments.

## 14. Prioritized recommendations

1. Use the verified result to define a controlled follow-up rather than an automatic decision.
2. Monitor the weakest subgroup, time window, interval coverage, or cluster sensitivity shown in the saved tables.
3. Revalidate against a transparent baseline whenever the data or operating context changes.

## 15. Limitations, ethics, and responsible use

See project README.

Automated outputs remain associative unless a causal study design says otherwise.

## 16. Saved-artifact integrity

In [9]:
rows=[]
for path in sorted(REPORTS_DIR.rglob("*")):
    if path.is_file(): rows.append({"artifact": str(path.relative_to(PROJECT_ROOT)), "size_kb": round(path.stat().st_size/1000,1), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:12]})
artifacts=pd.DataFrame(rows); print(artifacts.to_string(index=False,max_rows=80))

                                    artifact  size_kb       sha256
reports\figures\gem_price_model_evidence.png    415.3 dca699a8810f
          reports\figures\test_residuals.png     52.8 f0dbd9651df9
                        reports\metrics.json      4.4 07f6c6abe8b1
             reports\tables\data_quality.csv      0.4 de68a4348fad
      reports\tables\error_by_carat_band.csv      0.4 1b9d63408ee3
         reports\tables\model_comparison.csv      0.5 08f516dd1fc0
   reports\tables\permutation_importance.csv      0.4 7110e86c5d72
         reports\tables\test_predictions.csv    428.8 8e5936d8857f


## 17. Acceptance check

In [10]:
metrics=json.loads((REPORTS_DIR/"metrics.json").read_text(encoding="utf-8"))
assert metrics["status"]=="passed"
assert list((REPORTS_DIR/"figures").glob("*.png"))
assert list((REPORTS_DIR/"tables").glob("*.csv"))
assert all(path.stat().st_size>0 for path in REPORTS_DIR.rglob("*") if path.is_file())
print("PASS: metrics status, figures, tables, and non-empty artifacts verified")

PASS: metrics status, figures, tables, and non-empty artifacts verified


## 18. Conclusion

The project addressed compare interpretable and nonlinear price models, diagnose residual risk, and quantify empirical prediction coverage. using leakage-safe imputation/encoding, median baseline, linear and regularized log-target models, random forest, histogram boosting, cross-validation, residual diagnostics, permutation importance, segment errors, and calibration-residual intervals. The final verified conclusion is: **Histogram gradient boosting reaches untouched-test R² 0.981, RMSE 563, and 90.6% coverage for the training-calibrated 90% diagnostic interval.** The next responsible step is external or current-data validation before operational use.

## 19. Reproduce locally

```bash
python projects/05-gem-price-regression/src/analysis.py
python scripts/execute_notebooks.py --project 05-gem-price-regression
```